In [1]:
"""
a. Create a Rest API in python using flask which supports the following operations on the
Northwind dataset
1. Insert, update and select on customers
2. Insert, update and select on products
3. Insert, update and select on orders
4. Order history of the given customer
"""
%pip install flask
%pip install sqlalchemy
%pip install Flask-SQLAlchemy
%pip install PyMysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:

from flask import Flask
from flask_sqlalchemy import SQLAlchemy
from sqlalchemy import URL, Text, LargeBinary

app = Flask(__name__)

url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="Bhak@13",  
    host="localhost",
    port=3306,
    database="northwind"
)

app.config["SQLALCHEMY_DATABASE_URI"] = url

db = SQLAlchemy(app)

In [7]:
class Products(db.Model):
    __tablename__="products"
    id = db.Column(db.Integer, primary_key=True, autoincrement=True)
    product_code = db.Column(db.String(25))
    product_name = db.Column(db.String(50))
    description = db.Column(Text)
    standard_cost = db.Column(db.Numeric(19, 4))
    list_price = db.Column(db.Numeric(19, 4), nullable=False)
    reorder_level = db.Column(db.Integer)
    target_level = db.Column(db.Integer)
    quantity_per_unit = db.Column(db.String(50))
    discontinued = db.Column(db.Boolean, nullable=False, default=False)
    minimum_reorder_quantity = db.Column(db.Integer)
    category = db.Column(db.String(50))
    supplier_ids = db.Column(Text)
    attachments = db.Column(LargeBinary)

with app.app_context():
    with db.engine.connect() as c:
        print("Succesfull")


Succesfull


In [8]:

@app.route("/products",methods=["GET"])
def get_products():
    cus=Products.query.all()
    return cus



In [ ]:
from flask import request
@app.route("/products",methods=["POST"])
def add_product():
    data=request.get_json()

    p = Products(
        product_code=data.get("product_code"),
        product_name=data.get("product_name"),
        description=data.get("description"),
        standard_cost=data.get("standard_cost"),
        list_price=data.get("list_price"),
        reorder_level=data.get("reorder_level"),
        target_level=data.get("target_level"),
        quantity_per_unit=data.get("quantity_per_unit"),
        discontinued=data.get("discontinued", False),
        minimum_reorder_quantity=data.get("minimum_reorder_quantity"),
        category=data.get("category"),
        supplier_ids=data.get("supplier_ids"),
        attachments=data.get("attachments")
    )

    db.session.add(p)
    db.session.commit()

    return {"message": "Customer added","id": p.id }, 201

In [ ]:
@app.route("/products/<int:id>",methods=["PUT"])
def update_products(id):
    p = db.session.get(Products,id)
    data=request.get_json()
    if p is None:
        return {"error": "Product not found"}, 404
    p.product_code = data.get("product_code", p.product_code)
    p.product_name = data.get("product_name", p.product_name)
    p.description = data.get("description", p.description)
    p.standard_cost = data.get("standard_cost", p.standard_cost)
    p.list_price = data.get("list_price", p.list_price)
    p.reorder_level = data.get("reorder_level", p.reorder_level)
    p.target_level = data.get("target_level", p.target_level)
    p.quantity_per_unit = data.get("quantity_per_unit", p.quantity_per_unit)
    p.discontinued = data.get("discontinued", p.discontinued)
    p.minimum_reorder_quantity = data.get("minimum_reorder_quantity", p.minimum_reorder_quantity)

    db.session.commit()
    
    return {"message": "Customer updated","id": p.id }, 201
    
    

In [10]:
@app.route("/products/name/<string:name>", methods=["GET"])
def get_product_by_name(name):
    p = Products.query.filter_by(product_name=name).first()

    if p is None:
        return {"error": "Product not found"}, 404
    return p